# Tree Methods Consulting Project - Solution

#### Task
You have been hired by a dog food company to identify why some batches spoil much faster than expected.

The company mixes four preservatives (`A`, `B`, `C`, `D`) and then adds a filler chemical. Due to older machinery, preservative percentages vary more than they should. Food scientists suspect one of `A`, `B`, `C`, or `D` is driving early spoilage.

Use a tree-based model to determine which preservative has the strongest predictive power for spoilage.

- `Pres_A`: percentage of preservative A in the mix
- `Pres_B`: percentage of preservative B in the mix
- `Pres_C`: percentage of preservative C in the mix
- `Pres_D`: percentage of preservative D in the mix
- `Spoiled`: label indicating whether the batch was spoiled

This is a feature-importance investigation task, not a typical train/test workflow.

In [0]:
from pyspark.sql import SparkSession
from sklearn.tree import DecisionTreeClassifier

#### Start Spark session and load dataset.

In [0]:
spark = SparkSession.builder.appName('dog_food_spoilage').getOrCreate()
dog_food_df = spark.read.csv('data/dog_food.csv', inferSchema=True, header=True)

In [0]:
dog_food_df.printSchema()

root
 |-- A: integer (nullable = true)
 |-- B: integer (nullable = true)
 |-- C: double (nullable = true)
 |-- D: integer (nullable = true)
 |-- Spoiled: double (nullable = true)



In [0]:
dog_food_df.head()

Row(A=4, B=2, C=12.0, D=3, Spoiled=1.0)

In [0]:
dog_food_df.describe().show()

+-------+----------------+------------------+------------------+-----------------+------------------+
|summary|               A|                 B|                 C|                D|           Spoiled|
+-------+----------------+------------------+------------------+-----------------+------------------+
|  count|             490|               490|               490|              490|               490|
|   mean|5.53469387755102| 5.504081632653061| 9.126530612244897|5.579591836734694|0.2857142857142857|
| stddev|2.95152042343991|2.8537966089662063|2.0555451971054146|2.854836930998285|0.4522156316461308|
|    min|               1|                 1|               5.0|                1|               0.0|
|    max|              10|                10|              14.0|               10|               1.0|
+-------+----------------+------------------+------------------+-----------------+------------------+



In [0]:
dog_food_df.columns

['A', 'B', 'C', 'D', 'Spoiled']

In [0]:
feature_columns = ['A', 'B', 'C', 'D']
spark_model_df = dog_food_df.select(*feature_columns, 'Spoiled')

#### Prepare features in Spark, then convert to pandas for sklearn modeling.

In [0]:
spark_model_df.show(5)

+---+---+----+---+-------+
|  A|  B|   C|  D|Spoiled|
+---+---+----+---+-------+
|  4|  2|12.0|  3|    1.0|
|  5|  6|12.0|  7|    1.0|
|  6|  2|13.0|  6|    1.0|
|  4|  2|12.0|  1|    1.0|
|  4|  2|12.0|  3|    1.0|
+---+---+----+---+-------+
only showing top 5 rows


In [0]:
pandas_model_df = spark_model_df.toPandas()

In [0]:
X = pandas_model_df[feature_columns]
y = pandas_model_df['Spoiled']
X.head()

,A,B,C,D
0,4,2,12.0,3
1,5,6,12.0,7
2,6,2,13.0,6
3,4,2,12.0,1
4,4,2,12.0,3


In [0]:
decision_tree = DecisionTreeClassifier(random_state=42)

#### Fit sklearn decision tree and inspect feature importances to identify the preservative with the strongest spoilage signal.

In [0]:
decision_tree_model = decision_tree.fit(X, y)

In [0]:
feature_importance = dict(zip(feature_columns, decision_tree_model.feature_importances_))
feature_importance

{'A': 0.010100169191078276,
 'B': 0.00860364064355194,
 'C': 0.9377980582890623,
 'D': 0.04349813187630763}

#### Conclusion
Feature importance indicates that **chemical C** (feature index `2`) is the strongest driver of early spoilage.